# Notebook 1 — Alur PCV Klasik Adaptif (Preprocessing → Segmentasi `if-else` → Ekstraksi Fitur → Bounding Box Manual)

Notebook Google Colab ini memvisualisasikan pipeline **Pengolahan Citra & Visi Komputer (PCV) Klasik** secara bertahap dan transparan pada citra kendaraan, dari gambar mentah hingga menghasilkan **Bounding Box manual tanpa label kelas**.

### Fitur Utama Notebook Ini
1. **Input Citra** — Unggah langsung lewat Colab atau ambil dari Google Drive (disarankan minimal 2 gambar: *Kasus Ideal* dan *Kasus Menantang*).
2. **Analisis Kondisi Citra & Aturan `if-else` Adaptif** — Secara otomatis membaca statistik piksel gambar (*pencahayaan*, *keramaian/tekstur background*, dan *kontras warna HSV*) untuk memilih parameter preprocessing & metode segmentasi terbaik, dilengkapi **validasi kualitas mask & fallback otomatis** (tetap bisa dialihkan ke *Mode Manual* via `AUTO_ADAPTIVE = False`).
3. **Preprocessing** — Konversi Grayscale, Gaussian Blur, CLAHE, Gamma Correction, dan perbandingan Histogram Intensitas.
4. **Segmentasi Klasik** — Perbandingan 4 metode (*Otsu*, *Adaptive Threshold*, *Canny Edge + Morfologi*, *HSV Saturation*) + evaluasi aturan validasi `if-else`.
5. **Ekstraksi Fitur** — Mengubah objek hasil segmentasi menjadi angka & visual:
   - **Fitur Bentuk / Geometri** (`Area`, `Perimeter`, `x, y, w, h`, `Aspect Ratio`, `Extent`, `Solidity`, `Circularity`, `Hu Moments`)
   - **Fitur Gradien Bentuk (HOG)**
   - **Fitur Tekstur (GLCM)** & **Ringkasan Warna RGB/HSV**
6. **Hasil Akhir** — **Bounding Box Manual Tanpa Label** (`cv2.boundingRect`), ringkasan alur 5-panel siap tempel ke PowerPoint (PNG 300 DPI), dan evaluasi multi-gambar adaptif.

## Cell 1 — Setup Library & Folder Output Presentasi

In [ ]:
import math
import zipfile
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from skimage.feature import graycomatrix, graycoprops, hog

try:
    from google.colab import drive, files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    display  # type: ignore[name-defined]
except NameError:
    display = print

OUT_DIR = Path("/content/outputs_presentasi/nb1_pcv_klasik") if IN_COLAB else Path("./outputs_presentasi/nb1_pcv_klasik")
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.titleweight"] = "bold"


def save_fig(fig, filename: str):
    path = OUT_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"Tersimpan: {path}")
    return path


print(f"Siap! Environment Colab={IN_COLAB} | OpenCV={cv2.__version__} | Output={OUT_DIR}")

## Cell 2 — Muat Gambar Uji (Upload Langsung atau dari Google Drive)

* Pilih `SOURCE_MODE = "upload"` untuk langsung mengunggah 1 atau beberapa gambar kendaraan dari laptop.
* Atau pilih `SOURCE_MODE = "drive"` dan arahkan `DRIVE_IMAGE_DIR` ke folder gambar di Google Drive.
* Ubah `ACTIVE_INDEX` (`0`, `1`, dst.) untuk mengganti gambar utama yang dibedah pada Cell 3–8.

In [ ]:
SOURCE_MODE = "upload"  # Pilihan: "upload" | "drive" | "local"
DRIVE_IMAGE_DIR = "/content/drive/MyDrive/ferqueue/sample_images"
LOCAL_IMAGE_DIR = "outputs/test-samples"
ACTIVE_INDEX = 0  # Indeks gambar utama yang dianalisis detail pada Cell 3-8

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
UPLOAD_DIR = OUT_DIR / "_inputs"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

image_paths = []
if SOURCE_MODE == "upload" and IN_COLAB:
    print("Silakan pilih 1 atau lebih gambar kendaraan (JPG/PNG)...")
    uploaded = colab_files.upload()
    for fname, content in uploaded.items():
        dst = UPLOAD_DIR / Path(fname).name
        dst.write_bytes(content)
        if dst.suffix.lower() in IMAGE_EXTS:
            image_paths.append(dst)
elif SOURCE_MODE == "drive" and IN_COLAB:
    drive.mount("/content/drive")
    src_dir = Path(DRIVE_IMAGE_DIR)
    image_paths = sorted(p for p in src_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
else:
    src_dir = Path(LOCAL_IMAGE_DIR)
    if src_dir.is_dir():
        image_paths = sorted(p for p in src_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)

if not image_paths:
    raise FileNotFoundError(
        "Belum ada gambar yang dimuat. Unggah minimal 1 gambar JPG/PNG atau periksa path folder."
    )


def load_rgb(path: Path) -> np.ndarray:
    with Image.open(path) as img:
        return np.asarray(ImageOps.exif_transpose(img).convert("RGB"), dtype=np.uint8)


loaded_images = [(p.name, load_rgb(p)) for p in image_paths]
ACTIVE_INDEX = min(ACTIVE_INDEX, len(loaded_images) - 1)
active_name, img_rgb = loaded_images[ACTIVE_INDEX]

print(f"Total gambar dimuat: {len(loaded_images)}")
for idx, (name, arr) in enumerate(loaded_images):
    marker = " <-- AKTIF" if idx == ACTIVE_INDEX else ""
    print(f"  [{idx}] {name} | ukuran={arr.shape[1]}x{arr.shape[0]} px | dtype={arr.dtype}{marker}")

cols = min(len(loaded_images), 4)
rows = math.ceil(len(loaded_images) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 3.6 * rows))
axes = np.atleast_1d(axes).ravel()
for i, ax in enumerate(axes):
    if i < len(loaded_images):
        name, arr = loaded_images[i]
        ax.imshow(arr)
        tag = " [AKTIF]" if i == ACTIVE_INDEX else ""
        ax.set_title(f"[{i}] {name[:24]}{tag}\n{arr.shape[1]}x{arr.shape[0]} px", fontsize=9)
    ax.axis("off")
fig.suptitle("Daftar Gambar Uji yang Dimuat", fontsize=13, fontweight="bold")
plt.tight_layout()
save_fig(fig, "00_daftar_gambar_uji.png")
plt.show()

## Cell 3 — Analisis Kondisi Citra & Aturan `if-else` Adaptif (Berbasis Statistik Piksel)

Alih-alih memakai 1 set angka parameter tetap untuk semua gambar, sel ini membaca **karakteristik piksel gambar** secara otomatis lalu menerapkan **3 aturan `if-else`**:
1. **Aturan Pencahayaan (`mean_gray`)**:
   - `< 85` (Gelap/Bayangan) → `gamma=0.75`, `clahe_clip=3.0`
   - `> 180` (Silau/Terang) → `gamma=1.35`, `clahe_clip=1.5`
   - Normal → `gamma=1.0`, `clahe_clip=2.0`
2. **Aturan Keramaian Background (`border_edge_density`)**:
   - `> 0.08` (Background ramai/bertekstur) → blur lebih kuat (`gaussian=9`), Canny lebih ketat, `morph_close=11`, `dilate_iter=1`
   - `<= 0.08` (Background relatif polos) → blur standar (`gaussian=5`), auto-Canny berbasis median, `morph_close=15`, `dilate_iter=2`
3. **Aturan Pemilihan Metode Utama (`center_sat`, `sat_diff`, `gray_contrast`)**:
   - Kendaraan berwarna cerah di latar netral → prioritaskan **`hsv`**
   - Kontras terang-gelap tajam di latar polos → prioritaskan **`otsu`**
   - Warna netral (putih/hitam/abu-abu) atau latar kompleks → prioritaskan **`canny_morph`**

> Ubah `AUTO_ADAPTIVE = False` jika ingin menguji parameter manual sendiri (di `MANUAL_PARAMS`).

In [ ]:
AUTO_ADAPTIVE = True  # True = gunakan aturan if-else otomatis | False = gunakan MANUAL_PARAMS di bawah

MANUAL_PARAMS = {
    "gaussian_kernel": 5,
    "clahe_clip": 2.5,
    "clahe_grid": 8,
    "gamma": 1.0,
    "canny_low": 40,
    "canny_high": 120,
    "dilate_iter": 2,
    "morph_close": 15,
    "morph_open": 5,
    "hsv_sat_thresh": 45,
    "preferred_method": "canny_morph",
}


def analyze_image_and_choose_rules(rgb: np.ndarray, auto: bool = True, manual_cfg: dict = None) -> dict:
    """Hitung statistik piksel citra dan tentukan parameter lewat aturan if-else."""
    h, w = rgb.shape[:2]
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)

    # Mask area tengah (50% tengah frame) vs pinggir (15% tepi frame)
    cy1, cy2 = int(0.25 * h), int(0.75 * h)
    cx1, cx2 = int(0.25 * w), int(0.75 * w)
    border_mask = np.ones((h, w), dtype=bool)
    by, bx = max(1, int(0.15 * h)), max(1, int(0.15 * w))
    border_mask[by : h - by, bx : w - bx] = False

    mean_gray = float(gray.mean())
    median_gray = float(np.median(gray))
    center_gray = float(gray[cy1:cy2, cx1:cx2].mean())
    border_gray = float(gray[border_mask].mean())
    gray_contrast = abs(center_gray - border_gray)

    sat = hsv[:, :, 1]
    center_sat = float(sat[cy1:cy2, cx1:cx2].mean())
    border_sat = float(sat[border_mask].mean())
    sat_diff = center_sat - border_sat

    # Kepadatan tepi di pinggir frame sebagai indikator keramaian/tekstur background
    quick_edges = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 60, 150)
    border_edge_density = float(np.mean(quick_edges[border_mask] > 0))

    stats = {
        "mean_gray": round(mean_gray, 1),
        "median_gray": round(median_gray, 1),
        "gray_contrast": round(gray_contrast, 1),
        "center_sat": round(center_sat, 1),
        "border_sat": round(border_sat, 1),
        "sat_diff": round(sat_diff, 1),
        "border_edge_density": round(border_edge_density, 4),
    }

    if not auto:
        cfg = dict(manual_cfg or MANUAL_PARAMS)
        cfg["stats"] = stats
        cfg["conditions"] = ["Mode Manual (aturan if-else dinonaktifkan)"]
        return cfg

    conditions = []

    # ATURAN 1: Pencahayaan -> GAMMA & CLAHE
    if mean_gray < 85:
        gamma, clahe_clip = 0.75, 3.0
        conditions.append(f"Pencahayaan Gelap (mean={mean_gray:.1f}<85) -> gamma=0.75, clahe=3.0")
    elif mean_gray > 180:
        gamma, clahe_clip = 1.35, 1.5
        conditions.append(f"Pencahayaan Terang/Silau (mean={mean_gray:.1f}>180) -> gamma=1.35, clahe=1.5")
    else:
        gamma, clahe_clip = 1.0, 2.0
        conditions.append(f"Pencahayaan Normal (mean={mean_gray:.1f}) -> gamma=1.0, clahe=2.0")

    # ATURAN 2: Keramaian Background -> Gaussian Blur, Canny, Morfologi
    if border_edge_density > 0.08:
        gaussian_k = 9
        canny_low = int(np.clip(0.75 * median_gray, 70, 130))
        canny_high = int(np.clip(1.85 * median_gray, 170, 240))
        morph_close, morph_open, dilate_iter = 11, 7, 1
        conditions.append(
            f"Background Ramai (edge_tepi={border_edge_density:.3f}>0.08) -> blur=9, Canny={canny_low}/{canny_high}, close=11"
        )
    else:
        gaussian_k = 5
        canny_low = int(np.clip(0.60 * median_gray, 35, 90))
        canny_high = int(np.clip(1.35 * median_gray, 100, 200))
        morph_close, morph_open, dilate_iter = 15, 5, 2
        conditions.append(
            f"Background Relatif Polos (edge_tepi={border_edge_density:.3f}<=0.08) -> blur=5, Canny={canny_low}/{canny_high}, close=15"
        )

    # ATURAN 3: Warna Kendaraan & Kontras -> Pilihan Metode Segmentasi Utama
    hsv_sat_thresh = int(np.clip(border_sat + 25, 35, 95))
    if center_sat > 65 and sat_diff > 25:
        preferred = "hsv"
        conditions.append(
            f"Kendaraan Berwarna Kontras (sat_tengah={center_sat:.1f}, selisih=+{sat_diff:.1f}) -> metode utama 'hsv' (th={hsv_sat_thresh})"
        )
    elif gray_contrast > 45 and border_edge_density <= 0.08:
        preferred = "otsu"
        conditions.append(
            f"Kontras Terang-Gelap Tinggi & Latar Polos (kontras={gray_contrast:.1f}) -> metode utama 'otsu'"
        )
    else:
        preferred = "canny_morph"
        conditions.append(
            f"Warna Netral / Latar Kompleks (sat_tengah={center_sat:.1f}, kontras={gray_contrast:.1f}) -> metode utama 'canny_morph'"
        )

    return {
        "gaussian_kernel": gaussian_k,
        "clahe_clip": clahe_clip,
        "clahe_grid": 8,
        "gamma": gamma,
        "canny_low": canny_low,
        "canny_high": canny_high,
        "dilate_iter": dilate_iter,
        "morph_close": morph_close,
        "morph_open": morph_open,
        "hsv_sat_thresh": hsv_sat_thresh,
        "preferred_method": preferred,
        "stats": stats,
        "conditions": conditions,
    }


rule_cfg = analyze_image_and_choose_rules(img_rgb, auto=AUTO_ADAPTIVE, manual_cfg=MANUAL_PARAMS)

print(f"=== HASIL ANALISIS KONDISI CITRA ({active_name}) ===")
for line in rule_cfg["conditions"]:
    print("  •", line)

df_rules = pd.DataFrame([
    {
        "Parameter": k,
        "Nilai_Terpilih": rule_cfg[k],
    }
    for k in (
        "preferred_method", "gaussian_kernel", "clahe_clip", "gamma",
        "canny_low", "canny_high", "dilate_iter", "morph_close", "morph_open", "hsv_sat_thresh",
    )
])
display(pd.DataFrame([rule_cfg["stats"]]))
display(df_rules)

## Cell 4 — Tahap 1: Preprocessing Citra (Menggunakan Parameter Hasil `if-else`)

Menjalankan rangkaian preprocessing sesuai parameter yang dipilih pada Cell 3:
1. **Grayscale** (1 channel intensitas `0–255`)
2. **Gaussian Blur** (meredam noise tekstur aspal/background)
3. **CLAHE + Gamma Correction** (mempertegas kontras lokal bodi kendaraan)
4. **Histogram Intensitas** sebelum vs sesudah preprocessing.

In [ ]:
def preprocess_classic(rgb: np.ndarray, cfg: dict) -> dict:
    ksize = int(cfg["gaussian_kernel"])
    clip = float(cfg["clahe_clip"])
    grid = int(cfg.get("clahe_grid", 8))
    gamma = float(cfg["gamma"])

    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (ksize, ksize), 0)
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid))
    enhanced = clahe.apply(blurred)
    if gamma != 1.0:
        lut = np.round(255.0 * (np.arange(256) / 255.0) ** gamma).astype(np.uint8)
        enhanced = cv2.LUT(enhanced, lut)

    rgb_blur = cv2.GaussianBlur(rgb, (ksize, ksize), 0)
    lab = cv2.cvtColor(rgb_blur, cv2.COLOR_RGB2LAB)
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    rgb_enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    return {
        "rgb": rgb,
        "gray": gray,
        "blurred": blurred,
        "enhanced": enhanced,
        "rgb_enhanced": rgb_enhanced,
    }


prep = preprocess_classic(img_rgb, rule_cfg)

fig = plt.figure(figsize=(16, 7.2))
gs = fig.add_gridspec(2, 4, height_ratios=[1.35, 1.0], hspace=0.35, wspace=0.18)

panels = [
    ("1. Citra Asli (RGB)", prep["rgb"], None),
    ("2. Grayscale (1 Channel)", prep["gray"], "gray"),
    (f"3. Gaussian Blur ({rule_cfg['gaussian_kernel']}x{rule_cfg['gaussian_kernel']})", prep["blurred"], "gray"),
    (f"4. CLAHE (clip={rule_cfg['clahe_clip']}, γ={rule_cfg['gamma']})", prep["enhanced"], "gray"),
]
for i, (title, im, cmap) in enumerate(panels):
    ax = fig.add_subplot(gs[0, i])
    ax.imshow(im, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")

ax_h1 = fig.add_subplot(gs[1, 0:2])
ax_h1.hist(prep["gray"].ravel(), bins=64, range=(0, 256), color="#4c72b0", alpha=0.85)
ax_h1.set_title("Histogram Intensitas Sebelum CLAHE (Grayscale)")
ax_h1.set_xlabel("Nilai Piksel (0 - 255)")
ax_h1.set_ylabel("Jumlah Piksel")
ax_h1.grid(alpha=0.25)

ax_h2 = fig.add_subplot(gs[1, 2:4])
ax_h2.hist(prep["enhanced"].ravel(), bins=64, range=(0, 256), color="#dd8452", alpha=0.85)
ax_h2.set_title("Histogram Intensitas Sesudah CLAHE + Gamma")
ax_h2.set_xlabel("Nilai Piksel (0 - 255)")
ax_h2.set_ylabel("Jumlah Piksel")
ax_h2.grid(alpha=0.25)

fig.suptitle(f"Tahap 1: Preprocessing PCV Klasik — {active_name}", fontsize=14, fontweight="bold")
save_fig(fig, "01_preprocessing.png")
plt.show()

## Cell 5 — Tahap 2: Segmentasi Klasik (4 Metode + Aturan 4: Validasi Mask & *Fallback* Otomatis)

Selain menjalankan 4 metode klasik (`otsu`, `adaptive`, `canny_morph`, `hsv`) plus varian ketat (`canny_strict`), sel ini menerapkan **Aturan 4 (`if-else` Validasi Kualitas Mask)**:
* Jika mask dari metode utama **bocor ke background** (`cakupan > 72%` atau kontur utama menyentuh keempat sisi frame sekaligus) → sistem otomatis melakukan *fallback* ke metode dengan skor kualitas terbaik!
* Jika mask **terlalu kecil / kosong** (`cakupan < 4%`) → sistem juga otomatis berpindah ke kandidat yang sehat.
* Skor kualitas mask menilai kewajaran luas (`5%–65%`), *solidity* bentuk, dan posisi objek yang tidak menempel penuh di keempat tepi frame.

In [ ]:
def auto_orient_foreground(binary_mask: np.ndarray) -> np.ndarray:
    h, w = binary_mask.shape
    border = np.concatenate([binary_mask[0, :], binary_mask[-1, :], binary_mask[:, 0], binary_mask[:, -1]])
    return cv2.bitwise_not(binary_mask) if np.mean(border > 0) > 0.5 else binary_mask


def fill_largest_regions(mask: np.ndarray, open_k: int = 5, close_k: int = 11) -> np.ndarray:
    k_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_k, open_k))
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_k, close_k))
    cleaned = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k_close)
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_OPEN, k_open)
    contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled = np.zeros_like(cleaned)
    min_area = 0.015 * mask.shape[0] * mask.shape[1]
    for cnt in contours:
        if cv2.contourArea(cnt) >= min_area:
            cv2.drawContours(filled, [cnt], -1, 255, thickness=cv2.FILLED)
    return filled


def evaluate_mask_quality(mask: np.ndarray) -> dict:
    """Hitung skor kewajaran mask untuk Aturan 4 (validasi & fallback if-else)."""
    h, w = mask.shape
    img_area = float(h * w)
    coverage = float(np.mean(mask > 0))
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return {"coverage": coverage, "healthy": False, "score": -1.0, "reason": "kosong"}

    largest = max(cnts, key=cv2.contourArea)
    area = float(cv2.contourArea(largest))
    area_ratio = area / img_area
    x, y, bw, bh = cv2.boundingRect(largest)
    touches = int(x <= 2) + int(y <= 2) + int(x + bw >= w - 3) + int(y + bh >= h - 3)
    hull_area = float(max(1.0, cv2.contourArea(cv2.convexHull(largest))))
    solidity = area / hull_area

    bbox_ratio = (bw * bh) / img_area
    if coverage > 0.72 or area_ratio > 0.72:
        return {"coverage": coverage, "healthy": False, "score": 0.1, "reason": f"bocor_background ({coverage*100:.0f}%)"}
    if touches == 4 or bbox_ratio > 0.85:
        return {"coverage": coverage, "healthy": False, "score": 0.15, "reason": f"bbox_penuh_frame ({bbox_ratio*100:.0f}%)"}
    if coverage < 0.04 or area_ratio < 0.03:
        return {"coverage": coverage, "healthy": False, "score": 0.2, "reason": f"terlalu_kecil ({coverage*100:.1f}%)"}

    # Skor ideal: area sekitar 15%-55%, solidity tinggi, tidak menempel banyak tepi
    cx, cy = x + bw / 2.0, y + bh / 2.0
    center_dist = math.hypot((cx - w / 2.0) / w, (cy - h / 2.0) / h)
    area_score = 1.0 - min(1.0, abs(area_ratio - 0.30) / 0.45)
    score = 0.45 * area_score + 0.35 * solidity + 0.20 * max(0.0, 1.0 - center_dist * 2.0) - 0.10 * max(0, touches - 1)
    return {"coverage": coverage, "healthy": True, "score": round(score, 4), "reason": "sehat"}


def run_adaptive_segmentations(prep_dict: dict, cfg: dict) -> tuple:
    enh = prep_dict["enhanced"]
    rgb_enh = prep_dict["rgb_enhanced"]
    ok, ck = int(cfg["morph_open"]), int(cfg["morph_close"])
    c_lo, c_hi = int(cfg["canny_low"]), int(cfg["canny_high"])
    d_it = int(cfg["dilate_iter"])
    hsv_th = int(cfg["hsv_sat_thresh"])

    # 1. Otsu
    _, otsu_raw = cv2.threshold(enh, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    otsu_raw = auto_orient_foreground(otsu_raw)
    otsu_mask = fill_largest_regions(otsu_raw, ok, ck)

    # 2. Adaptive
    adapt_raw = cv2.adaptiveThreshold(enh, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 25, 6)
    adapt_mask = fill_largest_regions(adapt_raw, ok, ck)

    # 3. Canny + Morfologi (utama)
    edges = cv2.Canny(enh, c_lo, c_hi)
    k_dil = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    canny_mask = fill_largest_regions(cv2.dilate(edges, k_dil, iterations=d_it), ok, ck)

    # 3b. Canny Ketat (cadangan otomatis bila Canny utama bocor ke background)
    edges_strict = cv2.Canny(enh, min(200, c_lo + 40), min(250, c_hi + 70))
    canny_strict_mask = fill_largest_regions(cv2.dilate(edges_strict, k_dil, iterations=1), max(5, ok), max(7, ck - 4))

    # 4. HSV Saturation
    hsv = cv2.cvtColor(rgb_enh, cv2.COLOR_RGB2HSV)
    sat_raw = (hsv[:, :, 1] >= hsv_th).astype(np.uint8) * 255
    hsv_mask = fill_largest_regions(sat_raw, ok, ck)

    results = {
        "otsu": {"label": "Otsu Thresholding", "raw": otsu_raw, "mask": otsu_mask},
        "adaptive": {"label": "Adaptive Thresholding", "raw": adapt_raw, "mask": adapt_mask},
        "canny_morph": {"label": "Canny Edge + Morfologi", "raw": edges, "mask": canny_mask},
        "hsv": {"label": f"HSV Saturation (th={hsv_th})", "raw": sat_raw, "mask": hsv_mask},
        "canny_strict": {"label": "Canny Ketat (Fallback)", "raw": edges_strict, "mask": canny_strict_mask},
    }
    for k, item in results.items():
        item["quality"] = evaluate_mask_quality(item["mask"])

    # ATURAN 4 (IF-ELSE VALIDASI & FALLBACK)
    pref = cfg["preferred_method"]
    pref_q = results[pref]["quality"]
    if pref_q["healthy"]:
        chosen_key = pref
        decision_log = f"LOLOS VALIDASI: Metode utama '{pref}' sehat (skor={pref_q['score']:.3f}, area={pref_q['coverage']*100:.1f}%)."
    else:
        ranked = sorted(results.items(), key=lambda kv: (kv[1]["quality"]["healthy"], kv[1]["quality"]["score"]), reverse=True)
        chosen_key, chosen_item = ranked[0]
        decision_log = (
            f"FALLBACK OTOMATIS: '{pref}' ditolak ({pref_q['reason']}) -> dialihkan ke '{chosen_key}' "
            f"(status={chosen_item['quality']['reason']}, skor={chosen_item['quality']['score']:.3f}, "
            f"area={chosen_item['quality']['coverage']*100:.1f}%)."
        )
    return results, chosen_key, decision_log


def make_overlay(rgb: np.ndarray, mask: np.ndarray, color=(40, 220, 90), alpha=0.45) -> np.ndarray:
    overlay = rgb.copy().astype(np.float32)
    tint = np.zeros_like(overlay)
    tint[:, :] = color
    m = mask > 0
    overlay[m] = (1 - alpha) * overlay[m] + alpha * tint[m]
    return overlay.astype(np.uint8)


seg_results, auto_selected_key, fallback_log = run_adaptive_segmentations(prep, rule_cfg)
print("=== KEPUTUSAN ATURAN 4 (VALIDASI MASK & FALLBACK) ===")
print(" ", fallback_log)

# Tabel kualitas semua kandidat segmentasi
q_rows = [
    {
        "Key": k,
        "Metode": v["label"],
        "Area_%": round(v["quality"]["coverage"] * 100, 1),
        "Status": v["quality"]["reason"],
        "Skor_Kualitas": v["quality"]["score"],
        "Dipilih_Otomatis": "YA" if k == auto_selected_key else "",
    }
    for k, v in seg_results.items()
]
display(pd.DataFrame(q_rows))

# Plot 4 metode utama (+ ganti kolom canny_morph jika fallback memilih canny_strict)
display_keys = ["otsu", "adaptive", "canny_strict" if auto_selected_key == "canny_strict" else "canny_morph", "hsv"]
fig, axes = plt.subplots(3, 4, figsize=(16, 10.5))
row_titles = ["Mask/Tepi Awal", "Setelah Morfologi + Fill", "Overlay pada Citra Asli"]

for col, key in enumerate(display_keys):
    res = seg_results[key]
    q = res["quality"]
    star = " ★ DIPILIH" if key == auto_selected_key else ""
    axes[0, col].imshow(res["raw"], cmap="gray")
    axes[0, col].set_title(f"{res['label']}{star}\n(key='{key}')", fontsize=9.5)
    axes[1, col].imshow(res["mask"], cmap="gray")
    axes[1, col].set_title(f"Area={q['coverage']*100:.1f}% | {q['reason']}", fontsize=9)
    axes[2, col].imshow(make_overlay(img_rgb, res["mask"]))
    axes[2, col].set_title(f"Overlay (skor={q['score']:.2f})", fontsize=9)
    for r in range(3):
        axes[r, col].axis("off")

for r, rt in enumerate(row_titles):
    axes[r, 0].text(
        -0.08, 0.5, rt, transform=axes[r, 0].transAxes,
        rotation=90, va="center", ha="right", fontsize=11, fontweight="bold",
    )

fig.suptitle(f"Tahap 2: Segmentasi Klasik + Validasi if-else — {active_name}", fontsize=14, fontweight="bold")
plt.tight_layout()
save_fig(fig, "02_perbandingan_segmentasi.png")
plt.show()

## Cell 6 — Penyempurnaan Mask Terpilih, Kontur & Isolasi Objek

Secara default, sel ini langsung memakai hasil terbaik dari aturan `if-else` (`auto_selected_key`).
Jika kamu tetap ingin memaksa mencoba metode tertentu secara manual, cukup ubah `OVERRIDE_METHOD` (misal `"otsu"`, `"canny_morph"`, `"canny_strict"`, `"adaptive"`, atau `"hsv"`).

In [ ]:
OVERRIDE_METHOD = None   # Biarkan None untuk ikut pilihan if-else otomatis, atau isi string key metode
MIN_AREA_RATIO = 0.02    # Abaikan kontur noise < 2% luas gambar
MAX_AREA_RATIO = 0.85    # Abaikan kontur yang menelan hampir seluruh frame (> 85%)
KEEP_TOP_K_OBJECTS = 1   # Ubah ke 2 atau 3 jika dalam 1 gambar ada >1 kendaraan

SELECTED_METHOD = OVERRIDE_METHOD if OVERRIDE_METHOD in seg_results else auto_selected_key


def extract_valid_contours(mask: np.ndarray, min_ratio=0.02, max_ratio=0.85, top_k=1):
    h, w = mask.shape
    img_area = float(h * w)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    valid = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        ratio = area / img_area
        if min_ratio <= ratio <= max_ratio:
            valid.append((area, cnt))
    valid.sort(key=lambda item: item[0], reverse=True)
    return [cnt for _, cnt in valid[:top_k]]


chosen_raw = seg_results[SELECTED_METHOD]["raw"]
chosen_mask = seg_results[SELECTED_METHOD]["mask"]
chosen_label = seg_results[SELECTED_METHOD]["label"]
valid_contours = extract_valid_contours(chosen_mask, MIN_AREA_RATIO, MAX_AREA_RATIO, KEEP_TOP_K_OBJECTS)

if not valid_contours:
    print("PERINGATAN: Tidak ada kontur lolos filter area. Mengambil kontur terbesar yang tersedia.")
    all_cnts, _ = cv2.findContours(chosen_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if all_cnts:
        valid_contours = [max(all_cnts, key=cv2.contourArea)]

selected_mask = np.zeros_like(chosen_mask)
cv2.drawContours(selected_mask, valid_contours, -1, 255, thickness=cv2.FILLED)

contour_vis = img_rgb.copy()
for cnt in valid_contours:
    hull = cv2.convexHull(cnt)
    cv2.drawContours(contour_vis, [cnt], -1, (0, 255, 80), 3)
    cv2.drawContours(contour_vis, [hull], -1, (255, 220, 0), 2)

cutout = cv2.bitwise_and(img_rgb, img_rgb, mask=selected_mask)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
axes[0].imshow(chosen_raw, cmap="gray")
axes[0].set_title(f"1. Raw ({chosen_label})")
axes[1].imshow(selected_mask, cmap="gray")
axes[1].set_title("2. Mask Objek Terpilih")
axes[2].imshow(contour_vis)
axes[2].set_title("3. Kontur (Hijau) & Convex Hull (Kuning)")
axes[3].imshow(cutout)
axes[3].set_title("4. Isolasi Foreground (Cutout)")
for ax in axes:
    ax.axis("off")

fig.suptitle(f"Penyempurnaan Segmentasi ({chosen_label}) — {len(valid_contours)} Objek Terdeteksi", fontsize=13, fontweight="bold")
plt.tight_layout()
save_fig(fig, "03_segmentasi_terpilih.png")
plt.show()

## Cell 7 — Tahap 3: Ekstraksi Fitur (Geometri Kontur, HOG, Tekstur GLCM, & Warna)

Mengubah objek hasil segmentasi menjadi **vektor/tabel angka** dan **visualisasi fitur**:
1. **Fitur Bentuk / Geometri**: `Area`, `Perimeter`, `x, y, w, h`, `Aspect Ratio (w/h)`, `Extent`, `Solidity`, `Circularity`, dan `Hu Moments`.
2. **Fitur Gradien Bentuk (HOG)**: menangkap pola orientasi tepi bodi & roda kendaraan.
3. **Fitur Tekstur (GLCM)**: `Contrast`, `Dissimilarity`, `Homogeneity`, `Energy`, `Correlation`.
4. **Fitur Warna (RGB/HSV)**: dicatat sebagai pembanding (kurang diskriminatif untuk jenis kendaraan karena warna kendaraan sangat bervariasi).

In [ ]:
def extract_all_features(rgb: np.ndarray, gray: np.ndarray, mask: np.ndarray, cnt: np.ndarray, obj_id: int = 1):
    area = float(cv2.contourArea(cnt))
    perimeter = float(cv2.arcLength(cnt, True))
    x, y, w, h = cv2.boundingRect(cnt)
    bbox_area = float(max(1, w * h))
    aspect_ratio = w / float(max(1, h))
    extent = area / bbox_area
    hull = cv2.convexHull(cnt)
    hull_area = float(max(1.0, cv2.contourArea(hull)))
    solidity = area / hull_area
    circularity = (4.0 * math.pi * area) / (perimeter ** 2) if perimeter > 0 else 0.0

    hu = cv2.HuMoments(cv2.moments(cnt)).flatten()
    hu_log = [-math.copysign(1.0, val) * math.log10(abs(val)) if abs(val) > 1e-12 else 0.0 for val in hu[:3]]

    roi_rgb = rgb[y : y + h, x : x + w]
    roi_gray = gray[y : y + h, x : x + w]
    roi_mask = mask[y : y + h, x : x + w]

    roi_128 = cv2.resize(roi_gray, (128, 128), interpolation=cv2.INTER_AREA)
    hog_vec, hog_vis = hog(
        roi_128,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        visualize=True,
    )

    roi_q = (roi_128 // 8).astype(np.uint8)
    glcm = graycomatrix(
        roi_q,
        distances=[1, 2],
        angles=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4],
        levels=32,
        symmetric=True,
        normed=True,
    )
    glcm_props = {
        prop: float(np.mean(graycoprops(glcm, prop)))
        for prop in ("contrast", "dissimilarity", "homogeneity", "energy", "correlation")
    }

    fg_rgb = roi_rgb[roi_mask > 0]
    if fg_rgb.size == 0:
        fg_rgb = roi_rgb.reshape(-1, 3)
    hsv_roi = cv2.cvtColor(roi_rgb, cv2.COLOR_RGB2HSV)
    fg_hsv = hsv_roi[roi_mask > 0]
    if fg_hsv.size == 0:
        fg_hsv = hsv_roi.reshape(-1, 3)

    row = {
        "Objek": obj_id,
        "BBox_x": x,
        "BBox_y": y,
        "BBox_w": w,
        "BBox_h": h,
        "Area_px2": round(area, 1),
        "Perimeter_px": round(perimeter, 1),
        "Aspect_Ratio": round(aspect_ratio, 3),
        "Extent": round(extent, 3),
        "Solidity": round(solidity, 3),
        "Circularity": round(circularity, 3),
        "Hu1_log": round(hu_log[0], 3),
        "Hu2_log": round(hu_log[1], 3),
        "HOG_Dimensi": int(hog_vec.size),
        "GLCM_Contrast": round(glcm_props["contrast"], 3),
        "GLCM_Dissimilarity": round(glcm_props["dissimilarity"], 3),
        "GLCM_Homogeneity": round(glcm_props["homogeneity"], 3),
        "GLCM_Energy": round(glcm_props["energy"], 3),
        "GLCM_Correlation": round(glcm_props["correlation"], 3),
        "RGB_Mean": f"({fg_rgb[:,0].mean():.0f}, {fg_rgb[:,1].mean():.0f}, {fg_rgb[:,2].mean():.0f})",
        "HSV_Sat_Mean": round(float(fg_hsv[:, 1].mean()), 1),
    }
    artifacts = {
        "bbox": (x, y, w, h),
        "roi_rgb": roi_rgb,
        "roi_128": roi_128,
        "hog_vec": hog_vec,
        "hog_vis": hog_vis,
        "glcm_props": glcm_props,
        "fg_rgb": fg_rgb,
    }
    return row, artifacts


feature_rows = []
feature_artifacts = []
for idx, cnt in enumerate(valid_contours, start=1):
    row, art = extract_all_features(img_rgb, prep["enhanced"], selected_mask, cnt, obj_id=idx)
    feature_rows.append(row)
    feature_artifacts.append(art)

df_features = pd.DataFrame(feature_rows)
df_features.to_csv(OUT_DIR / "04_tabel_fitur.csv", index=False)

print("=== TABEL HASIL EKSTRAKSI FITUR (BENTUK, HOG, GLCM, WARNA) ===")
display(df_features.T.rename(columns=lambda c: f"Objek #{c + 1}"))

art0 = feature_artifacts[0]
print(f"\nDimensi vektor fitur HOG : {art0['hog_vec'].size} angka")
print("10 nilai pertama vektor HOG:", np.round(art0["hog_vec"][:10], 4))

fig = plt.figure(figsize=(16, 8.2))
gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.22)

ax1 = fig.add_subplot(gs[0, 0])
ax1.imshow(art0["roi_rgb"])
bx, by, bw, bh = art0["bbox"]
ax1.set_title(f"1. ROI Objek Hasil Segmentasi\n({bw}x{bh} px | AR={feature_rows[0]['Aspect_Ratio']})")
ax1.axis("off")

ax2 = fig.add_subplot(gs[0, 1])
ax2.imshow(art0["hog_vis"], cmap="inferno")
ax2.set_title(f"2. Visualisasi Fitur HOG\n(Vektor = {art0['hog_vec'].size} dimensi)")
ax2.axis("off")

ax3 = fig.add_subplot(gs[0, 2])
geom_labels = ["Aspect\nRatio", "Extent", "Solidity", "Circularity"]
geom_vals = [
    feature_rows[0]["Aspect_Ratio"],
    feature_rows[0]["Extent"],
    feature_rows[0]["Solidity"],
    feature_rows[0]["Circularity"],
]
bars = ax3.bar(geom_labels, geom_vals, color=["#4c72b0", "#55a868", "#c44e52", "#8172b2"])
for bar, val in zip(bars, geom_vals):
    ax3.text(bar.get_x() + bar.get_width() / 2, val + 0.03, f"{val:.2f}", ha="center", fontsize=9, fontweight="bold")
ax3.set_title("3. Fitur Geometri / Bentuk")
ax3.set_ylim(0, max(geom_vals) * 1.25 + 0.1)
ax3.grid(axis="y", alpha=0.25)

ax4 = fig.add_subplot(gs[1, 0:2])
glcm_keys = list(art0["glcm_props"].keys())
glcm_vals = [art0["glcm_props"][k] for k in glcm_keys]
bars_g = ax4.barh(glcm_keys, glcm_vals, color="#dd8452")
for bar, val in zip(bars_g, glcm_vals):
    ax4.text(val + max(glcm_vals) * 0.02, bar.get_y() + bar.get_height() / 2, f"{val:.3f}", va="center", fontsize=9)
ax4.set_title("4. Fitur Tekstur GLCM (Gray-Level Co-occurrence Matrix)")
ax4.set_xlim(0, max(glcm_vals) * 1.22 + 0.1)
ax4.grid(axis="x", alpha=0.25)

ax5 = fig.add_subplot(gs[1, 2])
for ch, col, lbl in zip(range(3), ["red", "green", "blue"], ["R", "G", "B"]):
    ax5.hist(art0["fg_rgb"][:, ch], bins=32, range=(0, 256), color=col, alpha=0.45, label=lbl)
ax5.set_title("5. Histogram Warna RGB Foreground\n(Kurang diskriminatif utk jenis kendaraan)")
ax5.legend()
ax5.grid(alpha=0.25)

fig.suptitle(f"Tahap 3: Ekstraksi Fitur PCV Klasik — {active_name}", fontsize=14, fontweight="bold")
save_fig(fig, "04_ekstraksi_fitur.png")
plt.show()

## Cell 8 — Tahap 4: Hasil Akhir Bounding Box Manual (Tanpa Label) & Diagram Alur 1-Baris (Siap PPT)

Menampilkan **Bounding Box manual tanpa label kelas** (`cv2.boundingRect`) dan **ringkasan alur 5-panel horizontal** siap tempel ke slide PowerPoint.

In [ ]:
SHOW_SIZE_TEXT = False  # Set True jika ingin menampilkan ukuran piksel (tetap tanpa label kelas)

bbox_vis = img_rgb.copy()
for art in feature_artifacts:
    x, y, w, h = art["bbox"]
    cv2.rectangle(bbox_vis, (x, y), (x + w, y + h), (40, 220, 80), 4)
    if SHOW_SIZE_TEXT:
        cv2.putText(
            bbox_vis, f"{w}x{h} px", (x + 4, max(22, y - 8)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (40, 220, 80), 2, cv2.LINE_AA,
        )

fig1, ax = plt.subplots(figsize=(7.5, 6))
ax.imshow(bbox_vis)
ax.set_title(f"Hasil Akhir PCV Klasik: Bounding Box Manual (Tanpa Label)\nMetode Terpilih (if-else): {chosen_label}")
ax.axis("off")
save_fig(fig1, "05_hasil_bounding_box.png")
plt.show()

fig2, axes = plt.subplots(1, 5, figsize=(20, 4.3))
summary_panels = [
    ("1. Input Citra", img_rgb, None),
    (f"2. Preprocessing\n(Blur={rule_cfg['gaussian_kernel']}, γ={rule_cfg['gamma']})", prep["enhanced"], "gray"),
    (f"3. Segmentasi (if-else)\n({chosen_label})", selected_mask, "gray"),
    ("4. Ekstraksi Fitur\n(HOG Gradien ROI)", feature_artifacts[0]["hog_vis"], "inferno"),
    ("5. Output Akhir\n(Bounding Box Manual)", bbox_vis, None),
]
for ax, (title, im, cmap) in zip(axes, summary_panels):
    ax.imshow(im, cmap=cmap)
    ax.set_title(title, fontsize=10.5)
    ax.axis("off")

fig2.suptitle(f"Ringkasan Alur PCV Klasik Adaptif — {active_name}", fontsize=14, fontweight="bold")
plt.tight_layout()
save_fig(fig2, "06_alur_lengkap_pcv_klasik.png")
plt.show()

## Cell 9 — Evaluasi Multi-Gambar Adaptif (`if-else` per Gambar) & Unduh Paket Gambar PPT

Setiap gambar uji dievaluasi **secara mandiri** oleh aturan `if-else`, sehingga gambar gelap, gambar terang, mobil berwarna cerah, dan mobil berwarna netral masing-masing mendapatkan parameter & metode yang berbeda secara otomatis!

Semua hasil PNG (300 DPI) + tabel CSV dikemas ke `nb1_pcv_klasik_outputs.zip`.

In [ ]:
rows_n = len(loaded_images)
fig, axes = plt.subplots(rows_n, 4, figsize=(16.5, 3.7 * rows_n))
if rows_n == 1:
    axes = np.expand_dims(axes, axis=0)

multi_summary = []
for r, (name, rgb_i) in enumerate(loaded_images):
    cfg_i = analyze_image_and_choose_rules(rgb_i, auto=AUTO_ADAPTIVE, manual_cfg=MANUAL_PARAMS)
    p_i = preprocess_classic(rgb_i, cfg_i)
    seg_i, key_i, log_i = run_adaptive_segmentations(p_i, cfg_i)
    if OVERRIDE_METHOD in seg_i:
        key_i = OVERRIDE_METHOD
    s_i = seg_i[key_i]["mask"]
    lbl_i = seg_i[key_i]["label"]
    cnts_i = extract_valid_contours(s_i, MIN_AREA_RATIO, MAX_AREA_RATIO, KEEP_TOP_K_OBJECTS)
    mask_i = np.zeros_like(s_i)
    vis_i = rgb_i.copy()
    if cnts_i:
        cv2.drawContours(mask_i, cnts_i, -1, 255, thickness=cv2.FILLED)
        for cnt in cnts_i:
            x, y, w, h = cv2.boundingRect(cnt)
            cv2.rectangle(vis_i, (x, y), (x + w, y + h), (40, 220, 80), 4)
        bx, by, bw, bh = cv2.boundingRect(cnts_i[0])
        ar = round(bw / float(max(1, bh)), 2)
    else:
        bw, bh, ar = 0, 0, 0.0

    multi_summary.append({
        "Gambar": name[:24],
        "Mean_Gray": cfg_i["stats"]["mean_gray"],
        "Edge_Tepi": cfg_i["stats"]["border_edge_density"],
        "Metode_Utama": cfg_i["preferred_method"],
        "Metode_Akhir": key_i,
        "Status": seg_i[key_i]["quality"]["reason"],
        "BBox_px": f"{bw}x{bh}",
        "Cakupan_%": round(100.0 * np.mean(mask_i > 0), 1),
    })

    axes[r, 0].imshow(rgb_i)
    axes[r, 0].set_title(f"Input: {name[:22]}\n(gray={cfg_i['stats']['mean_gray']}, sat={cfg_i['stats']['center_sat']})", fontsize=9)
    axes[r, 1].imshow(p_i["enhanced"], cmap="gray")
    axes[r, 1].set_title(f"Prep (blur={cfg_i['gaussian_kernel']}, γ={cfg_i['gamma']})", fontsize=9)
    axes[r, 2].imshow(mask_i, cmap="gray")
    axes[r, 2].set_title(f"Seg: {lbl_i}\n(area={100.0*np.mean(mask_i>0):.1f}%)", fontsize=9)
    axes[r, 3].imshow(vis_i)
    axes[r, 3].set_title(f"Bounding Box Manual ({len(cnts_i)} objek)", fontsize=9)
    for c in range(4):
        axes[r, c].axis("off")

fig.suptitle("Evaluasi Multi-Gambar: PCV Klasik Adaptif (Aturan if-else per Gambar)", fontsize=14, fontweight="bold")
plt.tight_layout()
save_fig(fig, "07_evaluasi_multi_gambar_adaptif.png")
plt.show()

df_multi = pd.DataFrame(multi_summary)
df_multi.to_csv(OUT_DIR / "07_ringkasan_aturan_if_else.csv", index=False)
display(df_multi)

zip_path = OUT_DIR.parent / "nb1_pcv_klasik_outputs.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT_DIR.glob("*")):
        if p.is_file():
            zf.write(p, p.name)
print(f"\nSemua gambar PPT tersimpan di: {zip_path} ({zip_path.stat().st_size / 1024:.1f} KB)")
# Hapus tanda '#' di bawah jika ingin langsung mengunduh ZIP di Colab:
# if IN_COLAB:
#     colab_files.download(str(zip_path))